In [15]:
import csv
from tqdm import tqdm
import os
import re

def normalize_option_id(option_id):
    """
    Remove extra whitespace and convert numeric strings to integer strings
    to avoid trailing '.0' (if the number is an integer).
    """
    # Remove whitespace
    normalized = re.sub(r'\s+', '', option_id)
    try:
        num = float(normalized)
        # If the number is an integer, convert it to an int string
        if num.is_integer():
            normalized = str(int(num))
        else:
            normalized = str(num)
    except ValueError:
        # If it's not a number, leave it as-is
        pass
    return normalized

current_dir = os.getcwd()
# File paths (update these as needed)
defval_file = os.path.join(current_dir, r"Helper Data\defval_ITA.csv")
input_file = os.path.join(current_dir, r"organized\1_2022_organized.csv")
output_file = os.path.join(current_dir, r"organized\2_2022_translated.csv")

# Step 1: Load the defval_ITA.csv into a dictionary for quick lookup,
# normalizing the keys
defval_mapping = {}
with open(defval_file, mode='r', encoding='ISO-8859-1') as defval_csv:
    reader = csv.DictReader(defval_csv)
    for row in reader:
        key = normalize_option_id(row['textid'])
        defval_mapping[key] = row['description']

# Step 2: Process the input file and generate the output file
with open(input_file, mode='r', encoding='ISO-8859-1') as infile, \
     open(output_file, mode='w', encoding='ISO-8859-1', newline='') as outfile:
    reader = csv.DictReader(infile)
    fieldnames = reader.fieldnames
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()

    rows = list(reader)  # Load rows to calculate total for progress bar
    for row in tqdm(rows, desc="Processing rows", unit="row"):
        # Process columns starting from index 3 (skipping anonid, office, obsscenario)
        for column in fieldnames[3:]:
            value = row[column].strip()
            # If the value contains "^", process each part separately
            if "^" in value:
                parts = value.split("^")
                updated_parts = []
                for part in parts:
                    part = part.strip()  # Ensure no extra spaces
                    if "$" in part:
                        id_rate = [p.strip() for p in part.split("$")]
                        # Normalize the ID part
                        clean_id = normalize_option_id(id_rate[0])
                        # Add prefix if not already present
                        textid = clean_id if clean_id.startswith("defval_") else f"defval_{clean_id}"
                        description = defval_mapping.get(textid, id_rate[0])
                        updated_parts.append(f"{description}${id_rate[1]}")
                    else:
                        clean_part = normalize_option_id(part)
                        textid = clean_part if clean_part.startswith("defval_") else f"defval_{clean_part}"
                        updated_parts.append(defval_mapping.get(textid, part))
                row[column] = "^".join(updated_parts)
            else:
                    # NEW: handle single "aspect$rate" case (no ^, but has $)
                 if "$" in value:
                    id_rate = [p.strip() for p in value.split("$", 1)]  # split only once
                    clean_id = normalize_option_id(id_rate[0])
                    textid = clean_id if clean_id.startswith("defval_") else f"defval_{clean_id}"
                    description = defval_mapping.get(textid, id_rate[0])
                    row[column] = f"{description}${id_rate[1]}" if len(id_rate) > 1 else description

                 else:
                    clean_value = normalize_option_id(value)
                    textid = clean_value if clean_value.startswith("defval_") else f"defval_{clean_value}"
                    row[column] = defval_mapping.get(textid, value)

        writer.writerow(row)  # Write the updated row to the output file

print(f"Processing completed. Output saved to {output_file}")


Processing rows: 100%|██████████| 1527/1527 [00:00<00:00, 2083.30row/s]

Processing completed. Output saved to C:\Users\Ramin\source\repos\Research Repo\ModeChoiceHybrid\Fastweb\Data\organized\2_2022_translated.csv
